In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 16


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 2.55338017642498
Epoch 2/100, Loss: 2.695490352809429
Epoch 3/100, Loss: 2.797163747251034
Epoch 4/100, Loss: 2.7398133724927902
Epoch 5/100, Loss: 2.3424980491399765
Epoch 6/100, Loss: 2.822334349155426
Epoch 7/100, Loss: 2.8525632172822952
Epoch 8/100, Loss: 2.6562384516000748
Epoch 9/100, Loss: 2.546669267117977
Epoch 10/100, Loss: 2.915204681456089
Epoch 11/100, Loss: 2.6510131135582924
Epoch 12/100, Loss: 2.5145634710788727
Epoch 13/100, Loss: 2.5083244666457176
Epoch 14/100, Loss: 2.7076574862003326
Epoch 15/100, Loss: 2.5864038169384003


Epoch 16/100, Loss: 2.533773384988308
Epoch 17/100, Loss: 2.736528903245926
Epoch 18/100, Loss: 2.489161938428879
Epoch 19/100, Loss: 2.7320399582386017
Epoch 20/100, Loss: 2.531670033931732
Epoch 21/100, Loss: 2.828640639781952
Epoch 22/100, Loss: 2.765851892530918
Epoch 23/100, Loss: 2.747167728841305
Epoch 24/100, Loss: 2.8095337599515915
Epoch 25/100, Loss: 2.9569828808307648
Epoch 26/100, Loss: 2.680164374411106
Epoch 27/100, Loss: 2.8049691393971443
Epoch 28/100, Loss: 2.898306220769882
Epoch 29/100, Loss: 2.760156363248825
Epoch 30/100, Loss: 2.827836751937866


Epoch 31/100, Loss: 2.7523029893636703
Epoch 32/100, Loss: 2.7423907071352005
Epoch 33/100, Loss: 2.797538012266159
Epoch 34/100, Loss: 2.7237603589892387
Epoch 35/100, Loss: 2.696680448949337
Epoch 36/100, Loss: 2.7173379212617874
Epoch 37/100, Loss: 2.835376352071762
Epoch 38/100, Loss: 2.6067269295454025
Epoch 39/100, Loss: 2.850962519645691
Epoch 40/100, Loss: 2.5342081859707832
Epoch 41/100, Loss: 2.7634591832756996
Epoch 42/100, Loss: 2.498912163078785
Epoch 43/100, Loss: 2.723916471004486
Epoch 44/100, Loss: 2.744577996432781
Epoch 45/100, Loss: 2.578026369214058


Epoch 46/100, Loss: 2.775046654045582
Epoch 47/100, Loss: 2.5988893657922745
Epoch 48/100, Loss: 2.726002410054207
Epoch 49/100, Loss: 2.6842555105686188
Epoch 50/100, Loss: 2.6797557175159454
Epoch 51/100, Loss: 2.489896535873413
Epoch 52/100, Loss: 2.8072312250733376
Epoch 53/100, Loss: 2.629721350967884
Epoch 54/100, Loss: 2.875641703605652
Epoch 55/100, Loss: 2.715119369328022
Epoch 56/100, Loss: 2.7696256563067436


Epoch 57/100, Loss: 2.664498634636402
Epoch 58/100, Loss: 2.7409841120243073
Epoch 59/100, Loss: 2.5336210057139397
Epoch 60/100, Loss: 2.6459884494543076
Epoch 61/100, Loss: 2.6599235236644745
Epoch 62/100, Loss: 2.588060364127159
Epoch 63/100, Loss: 2.8535437136888504
Epoch 64/100, Loss: 2.6448590755462646
Epoch 65/100, Loss: 2.6605181246995926
Epoch 66/100, Loss: 2.550495631992817
Epoch 67/100, Loss: 2.5923682749271393


...

Epoch 12/100, Loss: 3.1899686455726624
Epoch 13/100, Loss: 3.2994094863533974
Epoch 14/100, Loss: 3.021749347448349
Epoch 15/100, Loss: 3.013542316854
Epoch 16/100, Loss: 3.198282830417156
Epoch 17/100, Loss: 3.0943378061056137
Epoch 18/100, Loss: 3.306327745318413
Epoch 19/100, Loss: 3.0212747007608414
Epoch 20/100, Loss: 3.0985694527626038
Epoch 21/100, Loss: 2.950143873691559
Epoch 22/100, Loss: 3.2684327960014343
Epoch 23/100, Loss: 3.12808321416378
Epoch 24/100, Loss: 3.0174945816397667
Epoch 25/100, Loss: 2.9465440958738327
Epoch 26/100, Loss: 3.095221258699894
Epoch 27/100, Loss: 3.220550552010536
Epoch 28/100, Loss: 3.225690573453903
Epoch 29/100, Loss: 3.01111800968647


Epoch 30/100, Loss: 2.9856832176446915
Epoch 31/100, Loss: 3.091893121600151
Epoch 32/100, Loss: 3.054154135286808
Epoch 33/100, Loss: 3.014682114124298
Epoch 34/100, Loss: 3.1413410753011703
Epoch 35/100, Loss: 3.233639135956764
Epoch 36/100, Loss: 3.135672688484192
Epoch 37/100, Loss: 3.016893371939659
Epoch 38/100, Loss: 3.0361188799142838
Epoch 39/100, Loss: 3.100283995270729
Epoch 40/100, Loss: 3.158794641494751
Epoch 41/100, Loss: 3.044174924492836
Epoch 42/100, Loss: 3.1397349685430527
Epoch 43/100, Loss: 3.0048436000943184
Epoch 44/100, Loss: 2.9736141115427017
Epoch 45/100, Loss: 3.0367296636104584
Epoch 46/100, Loss: 3.040349170565605
Epoch 47/100, Loss: 3.200237289071083
Epoch 48/100, Loss: 3.158343121409416


Epoch 49/100, Loss: 3.1417464315891266
Epoch 50/100, Loss: 3.1333883181214333
Epoch 51/100, Loss: 3.0409176126122475
Epoch 52/100, Loss: 3.157706595957279
Epoch 53/100, Loss: 3.064804494380951
Epoch 54/100, Loss: 3.1635373681783676
Epoch 55/100, Loss: 3.2336739897727966
Epoch 56/100, Loss: 3.031351923942566
Epoch 57/100, Loss: 2.9957756623625755
Epoch 58/100, Loss: 3.141853243112564
Epoch 59/100, Loss: 3.0311252400279045
Epoch 60/100, Loss: 3.085537478327751
Epoch 61/100, Loss: 3.1011264994740486
Epoch 62/100, Loss: 2.923849791288376
Epoch 63/100, Loss: 2.9246416464447975
Epoch 64/100, Loss: 2.9919104874134064
Epoch 65/100, Loss: 2.98945552110672
Epoch 66/100, Loss: 3.136084273457527


Epoch 67/100, Loss: 2.9902343824505806
Epoch 68/100, Loss: 3.113886520266533
Epoch 69/100, Loss: 3.0051100850105286
Epoch 70/100, Loss: 2.927602842450142
Epoch 71/100, Loss: 3.0932881236076355
Epoch 72/100, Loss: 3.004216253757477
Epoch 73/100, Loss: 3.057995945215225
Epoch 74/100, Loss: 2.9932969510555267
Epoch 75/100, Loss: 3.100516200065613
Epoch 76/100, Loss: 3.0616241842508316
Epoch 77/100, Loss: 3.14070875197649
Epoch 78/100, Loss: 3.2086053490638733
Epoch 79/100, Loss: 3.0232267007231712
Epoch 80/100, Loss: 3.173859938979149
Epoch 81/100, Loss: 3.111284926533699
Epoch 82/100, Loss: 3.0898988470435143
Epoch 83/100, Loss: 3.1475241407752037


Epoch 84/100, Loss: 3.089424565434456
Epoch 85/100, Loss: 3.0281419903039932
Epoch 86/100, Loss: 3.1115743443369865
Epoch 87/100, Loss: 3.219314455986023
Epoch 88/100, Loss: 3.123009294271469
Epoch 89/100, Loss: 3.032406009733677
Epoch 90/100, Loss: 3.1806648075580597
Epoch 91/100, Loss: 3.0112043619155884
Epoch 92/100, Loss: 3.0510179698467255
Epoch 93/100, Loss: 3.1856153681874275
Epoch 94/100, Loss: 3.144874319434166
Epoch 95/100, Loss: 3.1256134063005447
Epoch 96/100, Loss: 2.9086572229862213
Epoch 97/100, Loss: 3.0703356117010117
Epoch 98/100, Loss: 3.009650945663452
Epoch 99/100, Loss: 3.1697809398174286
Epoch 100/100, Loss: 3.1314705163240433
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.4986
